# SpectraLoRA Inference Client

Predict Raman spectra from SMILES strings using the deployed Modal endpoint.

In [ ]:
!pip -q install modal

In [ ]:
import modal
import numpy as np
import matplotlib.pyplot as plt

# Connect to the deployed predictor
RamanPredictor = modal.Cls.from_name("raman-inference", "RamanPredictor")

In [ ]:
# --- Arcadia-style plotting ---
AMBER   = '#F28360'
AEGEAN  = '#5088C5'
MARINE  = '#8A99AD'
A_BLACK = '#09090A'
CHARCOAL = '#484B50'
SEAWEED  = '#3B9886'

def arcadia_style(ax):
    for sp in ('top', 'right'):
        ax.spines[sp].set_visible(False)
    for sp in ('bottom', 'left'):
        ax.spines[sp].set_color(A_BLACK)
    ax.tick_params(axis='both', which='both',
                   color=A_BLACK, labelsize=11, direction='out', pad=5)

def plot_spectrum(result, show_raw=True, figsize=(14, 5)):
    """Plot predicted Raman spectrum from inference result."""
    x = np.array(result['x_grid'])
    y_ref = np.array(result['spectrum_refined'])
    y_raw = np.array(result['spectrum_raw'])
    peaks = result['peaks']['positions_cm']
    peaks_int = result['peaks']['intensities']

    fig, ax = plt.subplots(1, 1, figsize=figsize, facecolor='white')

    if show_raw:
        ax.fill_between(x, y_raw, alpha=0.15, color=MARINE, label='Raw (pre-RefNet)')
        ax.plot(x, y_raw, color=MARINE, lw=0.8, alpha=0.5)

    ax.plot(x, y_ref, color=AMBER, lw=1.8, label='Refined')

    # Mark peaks
    for pos, inten in zip(peaks[:15], peaks_int[:15]):
        idx = np.argmin(np.abs(x - pos))
        ax.plot(pos, y_ref[idx], 'v', color=AEGEAN, markersize=5, alpha=0.7)

    ax.set_xlabel(r'Raman shift (cm$^{-1}$)', fontsize=13, fontweight='medium', color=A_BLACK)
    ax.set_ylabel('Normalized intensity', fontsize=13, fontweight='medium', color=A_BLACK)
    ax.set_title(
        f"{result['smiles']}   |   {result['n_atoms']} atoms, {result['n_modes']} modes   |   {result['timing']['total_s']:.2f}s",
        fontsize=12, fontweight='semibold', color=A_BLACK, loc='left', pad=10,
    )
    ax.set_xlim(400, 2000)
    ax.set_ylim(-0.02, 1.05)
    ax.legend(fontsize=10, frameon=False)
    arcadia_style(ax)
    fig.tight_layout()
    return fig, ax

def peak_table(result, n=15):
    """Print top peaks as a table."""
    peaks = result['peaks']
    print(f"{'#':>3s}  {'Position (cm-1)':>16s}  {'Intensity':>10s}")
    print('-' * 34)
    for i, (pos, inten) in enumerate(zip(peaks['positions_cm'][:n], peaks['intensities'][:n])):
        print(f"{i+1:3d}  {pos:16.1f}  {inten:10.4f}")

## Single molecule prediction

In [ ]:
# --- Predict a single molecule ---
SMILES = "c1ccccc1"  # benzene

result = RamanPredictor().predict.remote(SMILES)
print(f"Atoms: {result['n_atoms']}, Modes: {result['n_modes']}")
print(f"Timing: {result['timing']}")

In [ ]:
fig, ax = plot_spectrum(result)
plt.show()

In [ ]:
peak_table(result)

## Batch prediction

In [ ]:
# --- Batch: overlay multiple molecules ---
molecules = {
    'Benzene': 'c1ccccc1',
    'Ethanol': 'CCO',
    'Aspirin': 'CC(=O)Oc1ccccc1C(=O)O',
    'Caffeine': 'Cn1c(=O)c2c(ncn2C)n(C)c1=O',
}

predictor = RamanPredictor()
results = {}
for name, smi in molecules.items():
    print(f"Predicting {name} ({smi})...")
    results[name] = predictor.predict.remote(smi)
    print(f"  -> {results[name]['n_atoms']} atoms, {results[name]['timing']['total_s']:.2f}s")

In [ ]:
# --- Overlay plot ---
colors = [AMBER, AEGEAN, SEAWEED, '#B5699B']
fig, ax = plt.subplots(1, 1, figsize=(14, 6), facecolor='white')

for i, (name, res) in enumerate(results.items()):
    x = np.array(res['x_grid'])
    y = np.array(res['spectrum_refined'])
    ax.plot(x, y + i * 0.15, color=colors[i % len(colors)], lw=1.5, label=name)

ax.set_xlabel(r'Raman shift (cm$^{-1}$)', fontsize=13, fontweight='medium', color=A_BLACK)
ax.set_ylabel('Normalized intensity (offset)', fontsize=13, fontweight='medium', color=A_BLACK)
ax.set_title('SpectraLoRA Predictions', fontsize=14, fontweight='semibold', color=A_BLACK, loc='left', pad=10)
ax.set_xlim(400, 2000)
ax.legend(fontsize=11, frameon=False, loc='upper right')
arcadia_style(ax)
fig.tight_layout()
plt.show()

## Predicted vs Experimental — Ground Truth Comparison

Load measured Raman spectra from the experimental database, predict via the endpoint, and overlay.

In [ ]:
import pandas as pd
import ast

# Load experimental spectra
REPO_ROOT = ".."
spec_df = pd.read_csv(f"{REPO_ROOT}/ramanchembl_pipeline/experimental_data/raman_spectra_db.csv")

# Molecules with known SMILES and their experimental spectrum IDs
MOLECULES = {
    "Glycine":         {"smiles": "NCC(=O)O",                                      "exp_id": 29},
    "L-Alanine":       {"smiles": "C[C@@H](N)C(=O)O",                              "exp_id": 31},
    "L-Phenylalanine": {"smiles": "N[C@@H](Cc1ccccc1)C(=O)O",                      "exp_id": 35},
    "L-Tryptophan":    {"smiles": "N[C@@H](Cc1c[nH]c2ccccc12)C(=O)O",              "exp_id": 38},
    "L-Tyrosine":      {"smiles": "N[C@@H](Cc1ccc(O)cc1)C(=O)O",                   "exp_id": 39},
    "Adenine":         {"smiles": "c1nc(N)c2nc[nH]c2n1",                            "exp_id": 8},
    "Cytosine":        {"smiles": "Nc1cc[nH]c(=O)n1",                               "exp_id": 16},
    "Thymine":         {"smiles": "Cc1c[nH]c(=O)[nH]c1=O",                          "exp_id": 53},
    "Uracil":          {"smiles": "O=c1cc[nH]c(=O)[nH]1",                           "exp_id": 57},
    "Guanine":         {"smiles": "Nc1nc2[nH]cnc2c(=O)[nH]1",                       "exp_id": 30},
    "Ascorbic Acid":   {"smiles": "OC[C@@H](O)[C@H]1OC(=O)C(O)=C1O",               "exp_id": 11},
    "Citric Acid":     {"smiles": "OC(=O)CC(O)(CC(=O)O)C(=O)O",                     "exp_id": 14},
    "Pyruvate":        {"smiles": "CC(=O)C(=O)O",                                   "exp_id": 49},
    "Succinic Acid":   {"smiles": "OC(=O)CCC(=O)O",                                 "exp_id": 52},
    "Glycerol":        {"smiles": "OCC(O)CO",                                        "exp_id": 28},
    "Fumarate":        {"smiles": "OC(=O)/C=C/C(=O)O",                              "exp_id": 26},
    "Acetoacetate":    {"smiles": "CC(=O)CC(=O)O",                                   "exp_id": 6},
    "Estradiol":       {"smiles": "C[C@]12CC[C@H]3[C@@H](CCc4cc(O)ccc43)[C@@H]1CC[C@@H]2O", "exp_id": 199},
}

def load_experimental_spectrum(exp_id):
    """Load and normalize an experimental spectrum from the DB."""
    row = spec_df[spec_df["id"] == exp_id].iloc[0]
    wn = np.array(ast.literal_eval(row["wavenumbers"]), dtype=np.float64)
    intensity = np.array(ast.literal_eval(row["intensity"]), dtype=np.float64)
    intensity = np.clip(intensity, 0, None)
    intensity = intensity / (intensity.max() + 1e-12)
    return wn, intensity

print(f"Loaded {len(spec_df)} experimental spectra")
print(f"Comparison set: {len(MOLECULES)} molecules")

In [ ]:
# --- Run predictions for all molecules ---
predictor = RamanPredictor()
predictions = {}
for name, info in MOLECULES.items():
    print(f"Predicting {name}...", end=" ")
    pred = predictor.predict.remote(info["smiles"])
    predictions[name] = pred
    print(f"{pred['n_atoms']} atoms, {pred['timing']['total_s']:.1f}s")

print(f"\nDone: {len(predictions)} predictions")

In [ ]:
# --- Separate overlay plot for each molecule: Predicted vs Experimental ---
from scipy.signal import find_peaks

for name, info in MOLECULES.items():
    pred = predictions[name]
    exp_wn, exp_int = load_experimental_spectrum(info["exp_id"])

    x_pred = np.array(pred["x_grid"])
    y_pred = np.array(pred["spectrum_refined"])
    y_raw = np.array(pred["spectrum_raw"])

    # Determine shared x range
    xlo = max(400, exp_wn.min() - 50)
    xhi = min(2000, exp_wn.max() + 50)

    # Cosine similarity in the overlap region
    exp_interp = np.interp(x_pred, exp_wn, exp_int, left=0, right=0)
    mask = (x_pred >= xlo) & (x_pred <= xhi)
    dot = np.dot(y_pred[mask], exp_interp[mask])
    cos_sim = dot / (np.linalg.norm(y_pred[mask]) * np.linalg.norm(exp_interp[mask]) + 1e-12)

    fig, ax = plt.subplots(1, 1, figsize=(14, 4.5), facecolor="white")

    # Experimental
    ax.plot(exp_wn, exp_int, color=A_BLACK, lw=1.8, label="Experimental", zorder=3)

    # Predicted refined
    ax.plot(x_pred, y_pred, color=AMBER, lw=1.6, label="Predicted (refined)", zorder=2)

    # Predicted raw as faint fill
    ax.fill_between(x_pred, y_raw, alpha=0.1, color=MARINE)

    ax.set_xlim(xlo, xhi)
    ax.set_ylim(-0.02, 1.08)
    ax.set_xlabel(r"Raman shift (cm$^{-1}$)", fontsize=13, fontweight="medium", color=A_BLACK)
    ax.set_ylabel("Normalized intensity", fontsize=13, fontweight="medium", color=A_BLACK)
    ax.set_title(
        f"{name}   |   {pred['n_atoms']} atoms   |   cos={cos_sim:.3f}   |   {pred['timing']['total_s']:.1f}s",
        fontsize=12, fontweight="semibold", color=A_BLACK, loc="left", pad=10,
    )
    ax.legend(fontsize=10, frameon=False, loc="upper right")
    arcadia_style(ax)
    fig.tight_layout()
    plt.show()